**1. Import**

In [17]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import hstack
import warnings
warnings.filterwarnings('ignore')

**2. Memanggil data yang sudah di scraping**

In [18]:
df = pd.read_csv('dataset.csv')
df.head()

,review,rating
0,mantap,5
1,Pengiriman murah gak bisa dipake kalo belum ja...,3
2,mntap,4
3,mau kirim barang lewat gosend lama banget nung...,1
4,"please fix yur bugs on change payment fiture, ...",3


**3. Preprocessing**

In [19]:
def clean_text(text):
  text = str(text).lower()

  # handle negasi & netral
  text = text.replace("tidak bagus", "tidak_bagus")
  text = text.replace("kurang bagus", "kurang_bagus")
  text = text.replace("tidak terlalu bagus", "tidak_terlalu_bagus")
  text = text.replace("biasa saja", "biasa_saja")

  text = re.sub(r'[^a-zA-A\s]', '', text)
  return text

df['clean_review'] = df['review'].apply(clean_text)

**4. Labeling**

In [20]:
def label_sentiment(rating, text):
  text = text.lower()

  if "biasa" in text or "lumayan" in text or "cukup" in text:
    return "Netral"

  if rating <= 2:
    return 'Negatif'
  elif rating == 3:
    return 'Netral'
  else:
    return 'Positif'

df['sentiment'] = df.apply(lambda x: label_sentiment(x['rating'], x['clean_review']), axis=1)
df[['review', 'rating', 'sentiment']].head()

,review,rating,sentiment
0,mantap,5,Positif
1,Pengiriman murah gak bisa dipake kalo belum ja...,3,Netral
2,mntap,4,Positif
3,mau kirim barang lewat gosend lama banget nung...,1,Negatif
4,"please fix yur bugs on change payment fiture, ...",3,Netral


**5. Menampilkan sentimen berdasarkan label**

In [21]:
df['sentiment'].value_counts()

,count
sentiment,
Positif,6905
Negatif,4352
Netral,743


**6. Feature Extraction**

In [22]:
# word-Level TF-IDF
tfidf_word = TfidfVectorizer(
    max_features=12000,
    ngram_range=(1,3),
    min_df=3
)

# char-Level TF-IDF
tfidf_char = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3,5),
    max_features=8000
)

from scipy.sparse import hstack

X_word = tfidf_word.fit_transform(df['clean_review'])
X_char = tfidf_char.fit_transform(df['clean_review'])

X = hstack([X_word, X_char])
y = df['sentiment']

**7. Split**

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

**8. Skema 1**
TF-IDF + Logistic Regression (80:20)

In [24]:
model1 = LogisticRegression(max_iter=200, class_weight='balanced')
model1.fit(X_train, y_train)

y_pred1 = model1.predict(X_test)

acc1 = accuracy_score(y_test, y_pred1)
print("Akurasi Model 1:", acc1)
print(classification_report(y_test, y_pred1))

Akurasi Model 1: 0.8647222222222222
              precision    recall  f1-score   support

     Negatif       0.81      0.88      0.84      1247
      Netral       0.40      0.39      0.39       236
     Positif       0.96      0.91      0.93      2117

    accuracy                           0.86      3600
   macro avg       0.72      0.73      0.72      3600
weighted avg       0.87      0.86      0.87      3600



**9. Skema 2**
TF-IDF + SVM (80:20)

In [25]:
model2 = LinearSVC(class_weight='balanced')
model2.fit(X_train, y_train)

y_pred2 = model2.predict(X_test)

acc2 = accuracy_score(y_test, y_pred2)
print("Akurasi Model 2:", acc2)
print(classification_report(y_test, y_pred2))

Akurasi Model 2: 0.8675
              precision    recall  f1-score   support

     Negatif       0.80      0.88      0.84      1247
      Netral       0.51      0.27      0.35       236
     Positif       0.93      0.93      0.93      2117

    accuracy                           0.87      3600
   macro avg       0.75      0.69      0.71      3600
weighted avg       0.86      0.87      0.86      3600



**10. Skema 3**
TF-IDF Word + char + SVM (70:30)

In [26]:
from scipy.sparse import hstack

# word n-gram
tfidf_word = TfidfVectorizer(max_features=8000, ngram_range=(1,2))

# char n-gram
tfidf_char = TfidfVectorizer(analyzer='char', ngram_range=(3,5), max_features=5000)

X_word = tfidf_word.fit_transform(df['clean_review'])
X_char = tfidf_char.fit_transform(df['clean_review'])

X3 = hstack([X_word, X_char])

X_train3, X_test3, y_train3, y_test3 = train_test_split(X3, y, test_size=0.3, random_state=42)

model3 = LinearSVC(
    class_weight='balanced',
    C=2.5,
    max_iter=5000
)
model3.fit(X_train3, y_train3)

pred3 = model3.predict(X_test3)

acc3 = accuracy_score(y_test3, pred3)
print("Akurasi Skema 3:", acc3)

Akurasi Skema 3: 0.8575


**11. Perbandingan Model**

In [27]:
print("Akurasi Model 1:", acc1)
print("Akurasi Model 2:", acc2)
print("Akurasi Model 3:", acc3)

Akurasi Model 1: 0.8647222222222222
Akurasi Model 2: 0.8675
Akurasi Model 3: 0.8575


**12. Inference**

In [28]:
text_test = [
    "aplikasi sangat membantu dan cepat",
    "driver sering cancel dan aplikasi error",
    "biasa saja tidak terlalu bagus"
]

# preprocessing
clean_test = [clean_text(t) for t in text_test]

# sequence
Xw = tfidf_word.transform(clean_test)
Xc = tfidf_char.transform(clean_test)

X_final = hstack([Xw, Xc])

# prediksi
pred = model3.predict(X_final)

# output
for t, h in zip(text_test, pred):
  print("Teks:", t)
  print("Sentimen:", h)
  print("-"*40)

Teks: aplikasi sangat membantu dan cepat
Sentimen: Positif
----------------------------------------
Teks: driver sering cancel dan aplikasi error
Sentimen: Positif
----------------------------------------
Teks: biasa saja tidak terlalu bagus
Sentimen: Netral
----------------------------------------


In [29]:
 !pip freeze > requirements.txt

In [30]:
from google.colab import files
files.download('requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>